Primero correr la siguiente celda para descargar una parte del dataset. Arrancamos con el sujeto 1 y el nervio mediano, para que no sea tan pesada la descarga. Tarda alrededor de 10 minutos. 

In [ ]:
import os
import openneuro

DATASET_PATH = os.path.join("..", "data", "raw", "ds004388")
SUBJECT_ID = "sub-001"
sub_path = os.path.join(DATASET_PATH, SUBJECT_ID)

if not os.path.exists(sub_path):
    print(f"Descargando {SUBJECT_ID}...")
    openneuro.download(
        dataset="ds004388",
        target_dir=DATASET_PATH,
        include=[f"{SUBJECT_ID}/", "dataset_description.json"],
    )
    print("¡Descarga completada!")
else:
    print(f"El sujeto {SUBJECT_ID} ya cuenta con datos en: {sub_path}")

Descargando sub-001...
🌍 Preparing to download ds004388 …
🔎 Verifying 66 of 68 files against NEMAR's checksums 
📥 Checking 68 files, downloading as needed (5 concurrent downloads). 


c:\Users\Ramiro Grin\AppData\Local\Programs\Python\Python312\Lib\site-packages\rich\live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

🔄 Request timed out while downloading sub-001/eeg/sub-001_task-median_run-05_events.json, retrying (5 retries 
remain, backing off 0.5s) …

In [ ]:
import os
import glob
import mne
import pandas as pd

# 1. Definir rutas relativas a la carpeta del sujeto
DATASET_PATH = os.path.join("..", "data", "raw", "ds004388")
SUBJECT_ID = "sub-001"
eeg_dir = os.path.join(DATASET_PATH, SUBJECT_ID, "eeg")

print("=== VERIFICACIÓN Y LECTURA DE DATOS ===")

# 2. Listar archivos BIDS encontrados
if os.path.exists(eeg_dir):
    files = os.listdir(eeg_dir)
    print(f"\nSe encontraron {len(files)} archivos en {eeg_dir}:")
    for f in files[:10]:  # Mostrar los primeros 10 archivos
        print(f" - {f}")
    if len(files) > 10:
        print(f"   ... y {len(files) - 10} archivos más.")
else:
    raise FileNotFoundError(f"No se encontró la carpeta: {eeg_dir}")

# 3. Buscar el archivo de señal del nervio mediano (run-01) y su archivo de eventos
eeg_files = sorted(glob.glob(os.path.join(eeg_dir, "*task-median_run-01*.vhdr")))
if not eeg_files:
    # Intentar buscar formatos alternativos (.edf o .set) si el formato BIDS no usa BrainVision (.vhdr)
    eeg_files = sorted(glob.glob(os.path.join(eeg_dir, "*task-median_run-01*.*")))
    eeg_files = [f for f in eeg_files if f.endswith(('.vhdr', '.edf', '.set'))]

events_files = sorted(glob.glob(os.path.join(eeg_dir, "*task-median_run-01_events.tsv")))

if not eeg_files:
    print("\n[Aviso] No se encontró aún el run-01 de task-median. Buscando cualquier archivo de señal disponible...")
    eeg_files = [f for f in glob.glob(os.path.join(eeg_dir, "*.*")) if f.endswith(('.vhdr', '.edf', '.set'))]

# 4. Cargar la señal con MNE
raw_path = eeg_files[0]
print(f"\nCargando señal desde: {os.path.basename(raw_path)}")
raw = mne.io.read_raw(raw_path, preload=True)

# 5. Imprimir información técnica básica del registro
print("\n--- Información del Registro Raw ---")
print(f"Frecuencia de muestreo (sfreq): {raw.info['sfreq']} Hz")
print(f"Duración total del registro: {raw.times[-1]:.2f} segundos ({raw.times[-1]/60:.2f} min)")
print(f"Cantidad total de canales: {len(raw.ch_names)}")
print(f"Nombres de los primeros 15 canales: {raw.ch_names[:15]}")

# 6. Cargar y visualizar la tabla de marcas / eventos (_events.tsv)
if events_files:
    events_path = events_files[0]
    print(f"\nCargando tabla de eventos desde: {os.path.basename(events_path)}")
    df_events = pd.read_csv(events_path, sep="\t")
    print("\nPrimeras filas del archivo de eventos (Marcas de estimulación):")
    print(df_events.head(10))
    print("\nTipos de estímulos / ensayos presentes en este run:")
    if 'trial_type' in df_events.columns:
        print(df_events['trial_type'].value_counts())
    elif 'value' in df_events.columns:
        print(df_events['value'].value_counts())
else:
    print("\n[!] No se encontró el archivo _events.tsv para este run en la carpeta local.")